# Preprocesamiento inicial



In [1]:
import pandas as pd
import numpy as np

## 1. Carga del dataset

Se parte del dataset original utilizado en la auditoría de calidad (`spanish_houses.csv`), compuesto por 100.000 registros y 41 variables.

El objetivo de este notebook es aplicar las transformaciones necesarias para obtener una versión estructurada y consistente del conjunto de datos, que servirá como base para el análisis exploratorio y las posteriores etapas de modelado.

In [2]:
df = pd.read_csv("../data/raw/spanish_houses.csv")
df.shape

(100000, 41)

## 2. Corrección de valores sospechosos

In [ ]:
# Eliminamos los 183 registros identificados como cabeceras introducidas como datos
df = df[df["house_id"] != "house_id"].copy()

In [11]:
print("Dimensiones después de la limpieza:", df.shape)
print("Registros con house_id = 'house_id':", (df["house_id"] == "house_id").sum())

Dimensiones después de la limpieza: (99817, 41)
Registros con house_id = 'house_id': 0


### Corrección de registros defectuosos

Durante la auditoría se identificaron 183 registros que correspondían a cabeceras del fichero introducidas incorrectamente como observaciones. Estos registros se eliminan del conjunto de datos antes de continuar con el preprocesado.

Tras esta eliminación, el dataset queda compuesto por 99.817 registros y 41 variables. Se comprueba además que no permanece ningún registro cuyo `house_id` sea igual al nombre de la propia variable.

## 3. Conversión de variables numéricas almacenadas como 

Durante la auditoría se identificaron varias variables de naturaleza cuantitativa almacenadas como `object`. Para facilitar su análisis y tratamiento posterior, se transformarán a tipos numéricos.

Las variables afectadas son `bath_num`, `construct_date`, `m2_real`, `m2_useful`, `price` y `room_num`.

Antes de realizar la conversión se tratarán explícitamente los valores textuales que representan ausencia de información, como `sin baños` y `sin habitación`. Los valores que no puedan convertirse a formato numérico se convertirán en `NaN`.


In [12]:
numeric_as_text = [
    "bath_num",
    "construct_date",
    "m2_real",
    "m2_useful",
    "price",
    "room_num"
]

# Valores textuales que representan ausencia de información
missing_text_values = {
    "bath_num": ["sin baños"],
    "room_num": ["sin habitación"]
}

# Cambiamos los valores por NaN
for col, values in missing_text_values.items():
    df[col] = df[col].replace(values, np.nan)

In [13]:
# Convertimos a formato numérico
for col in numeric_as_text:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [14]:
df[numeric_as_text].dtypes

bath_num          float64
construct_date    float64
m2_real             int64
m2_useful         float64
price               int64
room_num          float64
dtype: object

In [17]:
# Dado que hemos reemplazado por NaN, se comprueban los porcentajes de ausentes
df[numeric_as_text].isna().sum()

bath_num            670
construct_date    67941
m2_real               0
m2_useful         47156
price                 0
room_num           1255
dtype: int64

### Conclusión

Las variables `bath_num`, `construct_date`, `m2_real`, `m2_useful`, `price` y `room_num` se convierten a tipos numéricos.

Los valores textuales `sin baños` y `sin habitación`, que representan ausencia de información, se transforman en `NaN`. El resto de valores que no puedan interpretarse como numéricos también se convierten en valores ausentes mediante `errors="coerce"`.

De este modo, las variables cuantitativas quedan preparadas para realizar operaciones estadísticas y análisis exploratorio sobre ellas.

## 4. Tratamiento de variables binarias

La auditoría mostró que las variables binarias presentan distintas representaciones de los mismos valores. En algunos casos aparecen valores como `"0"`, `"1"`, `"0.0"` o `"1.0"`, mientras que también se detectaron registros defectuosos en los que el valor coincide con el nombre de la propia variable.

Se normalizarán estas variables utilizando `0` y `1` como categorías válidas y `NaN` para los valores ausentes o no válidos.

In [18]:
binary_cols = [
    "air_conditioner",
    "balcony",
    "built_in_wardrobe",
    "chimney",
    "garden",
    "lift",
    "reduced_mobility",
    "storage_room",
    "swimming_pool",
    "terrace"
]

for col in binary_cols:
    df[col] = df[col].replace(col, np.nan)

In [19]:
# Convertimos a numérico

for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [20]:
# Comprobación
for col in binary_cols:
    print(f"{col}: {df[col].unique()}")

air_conditioner: [0 1]
balcony: [0 1]
built_in_wardrobe: [0 1]
chimney: [0 1]
garden: [1 0]
lift: [nan  1.  0.]
reduced_mobility: [0 1]
storage_room: [0 1]
swimming_pool: [0 1]
terrace: [1 0]


In [21]:
df[binary_cols].dtypes

air_conditioner        int64
balcony                int64
built_in_wardrobe      int64
chimney                int64
garden                 int64
lift                 float64
reduced_mobility       int64
storage_room           int64
swimming_pool          int64
terrace                int64
dtype: object

### Conclusión

Las variables binarias han quedado normalizadas utilizando `0` y `1` como únicos valores válidos.

Tras la transformación, nueve de las diez variables no presentan valores ausentes y contienen exclusivamente los valores `0` y `1`. La variable `lift` mantiene algunos valores `NaN`, correspondientes a la ausencia de información detectada durante la auditoría.

Por tanto, las variables binarias quedan correctamente homogeneizadas y preparadas para las siguientes etapas del análisis. El tratamiento de los valores ausentes de `lift` se decidirá posteriormente junto con el resto de variables con datos faltantes.

## 5. Tratamiento de registros duplicados

Durante la auditoría del conjunto de datos original se identificaron 168 registros completamente duplicados. Sin embargo, algunos de estos registros coincidían con las cabeceras introducidas como datos y fueron eliminados en el apartado anterior junto con los 183 registros defectuosos.

In [22]:
# Eliminamos los registros completamente duplicados
df = df.drop_duplicates().copy()

In [23]:
print("Dimensiones después de eliminar duplicados:", df.shape)
print("Duplicados restantes:", df.duplicated().sum())

Dimensiones después de eliminar duplicados: (99816, 41)
Duplicados restantes: 0


### Conclusión

Tras la eliminación de los registros defectuosos, únicamente permanece un registro completamente duplicado. Este registro se elimina, quedando un conjunto de datos de **99.816 registros y 41 variables**.

La comprobación posterior confirma que no existen duplicados exactos en el conjunto de datos resultante.

Los identificadores `house_id` repetidos se tratarán de forma independiente en el siguiente apartado, ya que su repetición no implica necesariamente que los registros sean duplicados.

## 6. Tratamiento de house_id

La variable `house_id` actúa como identificador de las viviendas y no representa una característica de la propiedad que deba utilizarse directamente como variable explicativa.

Durante la auditoría se detectaron identificadores que aparecen en más de un registro. Por este motivo, antes de decidir su tratamiento se comprueba si estas repeticiones corresponden a duplicados exactos o a registros diferentes asociados al mismo identificador.

La decisión adoptada será conservar `house_id` como identificador para facilitar la trazabilidad de los registros, pero excluirlo posteriormente de las variables utilizadas en el análisis predictivo.

In [45]:
# Hacemos una comprobación como la de la auditoria, algunos de los repetidos eran duplicados, pero pocos

house_id_counts = df["house_id"].value_counts()
repeated_house_ids = house_id_counts[house_id_counts > 1]

print("house_id repetidos:", len(repeated_house_ids))
print("Registros asociados a house_id repetidos:", repeated_house_ids.sum())

print("\nDistribución del número de apariciones:")
print(repeated_house_ids.value_counts().sort_index())

house_id repetidos: 5293
Registros asociados a house_id repetidos: 10586

Distribución del número de apariciones:
count
2    5293
Name: count, dtype: int64


In [46]:
# Vemos que aunque sean repetidos, los registros no son duplicados exactos.
repeated_records = df[df["house_id"].duplicated(keep=False)]

exact_duplicates = repeated_records.duplicated().sum()

print("Registros con house_id repetido:", len(repeated_records))
print("Duplicados exactos entre ellos:", exact_duplicates)
print("Registros con house_id repetido no idénticos:", len(repeated_records) - exact_duplicates)

Registros con house_id repetido: 10586
Duplicados exactos entre ellos: 0
Registros con house_id repetido no idénticos: 10586


In [ ]:
# Se analizan los primeros 10 house_id repetidos

for house_id in repeated_house_ids.index[:10]:
    # Nos quedamos con los 2 registros con el mismo id
    group = df[df["house_id"] == house_id]

    different_cols = []

    for col in df.columns:
        if col != "house_id":
            # Si en esa variable hay más de un valor diferente añadir columna
            if group[col].nunique(dropna=False) > 1:
                different_cols.append(col)

    print("\n" + "="*70)
    print(f"house_id: {house_id}")
    print("Variables diferentes:", different_cols)

    if different_cols:
        print(group[["house_id"] + different_cols].to_string(index=False))


house_id: 81365996
Variables diferentes: ['obtention_date']
house_id obtention_date
81365996     2019-03-29
81365996     2019-03-26

house_id: 84273475
Variables diferentes: ['obtention_date']
house_id obtention_date
84273475     2019-03-30
84273475     2019-03-26

house_id: 84345412
Variables diferentes: ['obtention_date']
house_id obtention_date
84345412     2019-03-30
84345412     2019-03-26

house_id: 84299564
Variables diferentes: ['ad_last_update', 'obtention_date']
house_id                     ad_last_update obtention_date
84299564 Anuncio actualizado el 27 de marzo     2019-03-30
84299564  Anuncio actualizado el 4 de marzo     2019-03-26

house_id: 83889966
Variables diferentes: ['ad_last_update', 'obtention_date']
house_id                     ad_last_update obtention_date
83889966 Anuncio actualizado el 27 de marzo     2019-03-30
83889966 Anuncio actualizado el 17 de enero     2019-03-26

house_id: 83886489
Variables diferentes: ['obtention_date']
house_id obtention_date
8388

In [ ]:
from collections import Counter

different_columns = Counter()

for house_id, group in repeated_records.groupby("house_id"):
    for col in df.columns:
        if col != "house_id" and group[col].nunique(dropna=False) > 1:
            different_columns[col] += 1

# Convertimos Counter() en una serie de pandas y lo ordenamos
pd.Series(different_columns).sort_values(ascending=False)

obtention_date                   5293
ad_last_update                    324
price                              42
ad_description                     35
loc_full                            9
energetic_certif                    7
loc_neigh                           6
heating                             4
floor                               4
loc_district                        4
m2_real                             3
lift                                3
garden                              3
terrace                             2
loc_city                            2
companies_prov_vs_national_%        2
population_prov_vs_national_%       2
number_of_companies_prov            2
renta_media_prov                    2
bath_num                            2
m2_useful                           2
garage                              2
population_prov                     2
storage_room                        1
loc_zone                            1
orientation                         1
balcony     

### Conclusión

La repetición de `house_id` no implica necesariamente que los registros sean duplicados. El análisis muestra que la principal diferencia entre registros con el mismo identificador se encuentra en `obtention_date`, que cambia en los 5.293 identificadores repetidos. También aparecen diferencias en `ad_last_update` en 324 casos y, en menor medida, en variables como `price` (42 casos) o `ad_description` (35 casos).

Por tanto, los registros asociados a un mismo `house_id` parecen corresponder en muchos casos a distintas capturas o versiones de un mismo anuncio, especialmente debido a las diferencias en las fechas de obtención y actualización. No se eliminarán automáticamente estos registros, ya que hacerlo podría suponer perder información temporal relevante, especialmente cambios en el precio.

El `house_id` se conservará como identificador, pero no se utilizará inicialmente como variable predictora. El tratamiento de las distintas observaciones asociadas a un mismo inmueble se decidirá posteriormente en función del objetivo del modelado y de la posible utilización de la dimensión temporal.

## 7. Tratamiento de valores ausentes

In [52]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_percentage = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "Valores ausentes": missing,
    "Porcentaje": missing_percentage
})

missing_summary

,Valores ausentes,Porcentaje
ground_size,99816,100.00
unfurnished,99353,99.54
kitchen,97787,97.97
loc_street,85685,85.84
heating,74285,74.42
construct_date,67941,68.07
orientation,60584,60.70
garage,59189,59.30
loc_neigh,56310,56.41
m2_useful,47156,47.24


In [53]:
print("Variables con valores ausentes:", len(missing_summary))
print("Registros con al menos un valor ausente:", df.isna().any(axis=1).sum())
print("Porcentaje de registros con algún ausente:", round(df.isna().any(axis=1).mean() * 100, 2), "%")

Variables con valores ausentes: 19
Registros con al menos un valor ausente: 99816
Porcentaje de registros con algún ausente: 100.0 %


### Conclusiones

Tras las transformaciones realizadas, se identifican 19 variables con valores ausentes, dos más que en la auditoría inicial debido a los `NaN` introducidos al transformar `room_num` y `bath_num`. La proporción de valores faltantes es muy desigual entre las variables, destacando `ground_size`, `unfurnished` y `kitchen`, con porcentajes de ausencia superiores al 97 % (siendo `ground_size` del 100 %), así como `loc_street`, `heating`, `construct_date`, `orientation` y `garage`, que presentan también niveles elevados de ausencia.

En esta etapa no se realiza ninguna imputación ni eliminación general de valores ausentes. Su tratamiento se decidirá posteriormente, durante el análisis exploratorio y la preparación definitiva de los datos, teniendo en cuenta la naturaleza de cada variable, su porcentaje de ausencia y su posible relevancia para el modelo.

## 8. Tratamiento de `renta_media_prov`

Durante la auditoría se detectó una inconsistencia en la variable `renta_media_prov`. Aunque los valores presentan aparentemente una distribución válida, se observó la coexistencia de valores como `21.613` y `19.818`, que no resultan coherentes con una renta media expresada en euros.

El análisis de los registros asociados a estos valores muestra que el problema no está relacionado con una provincia o zona concreta, sino con la representación de la variable. Se interpreta que el punto está siendo utilizado como separador de miles y no como separador decimal. Por tanto, valores como `21.613` se corresponden con `21.613 €` y deben transformarse a `21613`.

In [54]:
print("Valores antes de la corrección:")
print(df["renta_media_prov"].dropna().sort_values().unique()[:10])

Valores antes de la corrección:
[   21.613    22.221    22.339    22.803    22.822    23.532    28.514
    29.4   19818.    19889.   ]


In [55]:
print(df["renta_media_prov"].describe())
print("\nValores únicos:")
print(df["renta_media_prov"].value_counts(dropna=False).head(15))

count    59193.00000
mean     11871.86074
std       9978.42992
min         21.61300
25%         22.82200
50%      19818.00000
75%      19818.00000
max      21714.00000
Name: renta_media_prov, dtype: float64

Valores únicos:
renta_media_prov
NaN          40623
19818.000    23197
22.803        8311
21714.000     7678
29.400        5248
19889.000     3806
22.822        2861
22.221        2644
21.613        1944
23.532        1401
22.339        1218
28.514         885
Name: count, dtype: int64


In [ ]:
# Simplemente multiplicamos por 1000 los valores menores que 100
df["renta_media_prov"] = df["renta_media_prov"].where(
    df["renta_media_prov"] >= 100,
    df["renta_media_prov"] * 1000
)

In [60]:
# Comprobamos que ya no hay ningún valor anómalo
print(df["renta_media_prov"].describe())
print("\n")
print(df["renta_media_prov"].value_counts(dropna=False).head(15))
print("\nValores inferiores a 100:", (df["renta_media_prov"] < 100).sum())

count    59193.000000
mean     21918.412093
std       2798.934352
min      19818.000000
25%      19818.000000
50%      21714.000000
75%      22803.000000
max      29400.000000
Name: renta_media_prov, dtype: float64


renta_media_prov
NaN        40623
19818.0    23197
22803.0     8311
21714.0     7678
29400.0     5248
19889.0     3806
22822.0     2861
22221.0     2644
21613.0     1944
23532.0     1401
22339.0     1218
28514.0      885
Name: count, dtype: int64

Valores inferiores a 100: 0


### Conclusiones

Se corrige la inconsistencia detectada en `renta_media_prov`, interpretando los valores inferiores a 100 como cantidades expresadas en miles. De este modo, valores como `21,613` pasan a representarse como `21.613 €`.

Tras la transformación, todos los valores de la variable se encuentran en un rango coherente, entre 19.818 € y 29.400 €, y no permanecen valores inferiores a 100.

Los valores ausentes se mantienen como `NaN` para decidir posteriormente su tratamiento en función de los resultados del análisis exploratorio.

## 9. Comprobaciones finales

In [61]:
print("Dimensiones finales:", df.shape)
print("Registros duplicados:", df.duplicated().sum())

Dimensiones finales: (99816, 41)
Registros duplicados: 0


In [62]:
print(df.dtypes.value_counts())

object     20
int64      13
float64     8
Name: count, dtype: int64


In [64]:
print("Valores 'house_id' como cabecera:", (df["house_id"] == "house_id").sum())
print("Valores de renta inferiores a 100:", (df["renta_media_prov"] < 100).sum())
print("Duplicados exactos:", df.duplicated().sum())
print("House_id nulos:",df["house_id"].isna().sum())

Valores 'house_id' como cabecera: 0
Valores de renta inferiores a 100: 0
Duplicados exactos: 0
House_id nulos: 0


### Conclusiones

Las comprobaciones finales confirman que las transformaciones realizadas se han aplicado correctamente. El conjunto de datos queda compuesto por 99.816 registros y 41 variables, sin registros completamente duplicados ni cabeceras introducidas como observaciones.

Asimismo, las principales variables numéricas almacenadas inicialmente como texto han sido convertidas a tipos adecuados, las variables binarias presentan una representación homogénea y `renta_media_prov` ha quedado expresada en una escala coherente.

Los valores ausentes se mantienen sin imputar y los identificadores `house_id` repetidos se conservan para su consideración en etapas posteriores. El conjunto de datos queda así preparado para comenzar el análisis exploratorio.

## 10. Exportación del dataset preprocesado

Una vez finalizadas las transformaciones iniciales y realizadas las comprobaciones de consistencia, se exporta el conjunto de datos resultante para utilizarlo como punto de partida en las siguientes etapas del proyecto.

El fichero original se mantiene sin modificaciones en `data/raw/`, mientras que el resultado del preprocesado se almacena en `data/processed/`, garantizando la trazabilidad entre los datos originales y las transformaciones realizadas.

In [66]:
processed_path = "../data/processed/spanish_houses_preprocessed.csv"
df.to_csv(processed_path, index=False)
print(f"Dataset preprocesado exportado en: {processed_path}")

Dataset preprocesado exportado en: ../data/processed/spanish_houses_preprocessed.csv


In [ ]:
df_check = pd.read_csv(processed_path)
print("Dimensiones del archivo exportado:", df_check.shape)

Dimensiones del archivo exportado: (99816, 41)
